In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
from collections import defaultdict
import math
import os

np.random.seed(42)

# --- CARD VALUES ---
CARD_VALUES = {
    "2": 2, "3": 3, "4": 4, "5": 5, "6": 6,
    "7": 7, "8": 8, "9": 9, "10": 10,
    "J": 10, "Q": 10, "K": 10, "A": 11
}

SUITS = ["hearts", "diamonds", "clubs", "spades"]
DECK = [f"{value} of {suit}" for value in CARD_VALUES for suit in SUITS]

#Function that checks if we have a natural Blackjack (first 2 cards)
def check_natural_blackjack(hand):
    return len(hand) == 2 and calculate_hand_value(hand) == 21

def calculate_hand_value(hand):
    value = 0
    aces = 0
    for card in hand:
        rank = card.split()[0]
        val = CARD_VALUES[rank]
        if rank == "A":
            aces += 1
        value += val
    while value > 21 and aces:
        value -= 10
        aces -= 1
    return value

def calculate_count(player_hand, dealer_card, count):
    cards = player_hand + [dealer_card]
    for card in cards:
        rank = card.split()[0]
        val = CARD_VALUES[rank]
        if 2 <= val <= 6:
            count += 1
        elif val == 10 or rank == "A":
            count -= 1
    return count

def adjust_count(new_card, count):
    new_card_value = CARD_VALUES[new_card.split()[0]] #only one card can be drawn after doubling-down
    if 2 <= new_card_value <= 6:
        count += 1
    elif 10 <= new_card_value <= 11:
        count -= 1

    return count

def get_valid_actions(player_hand):
    valid_actions = [0, 1]  #Hit, Stay
    if len(player_hand) == 2:
        valid_actions.append(2)  #Allow Double-down only on 2 cards
    return valid_actions

def play_dealer_card_counting(deck, dealer_hand, count):
    while calculate_hand_value(dealer_hand) < 17:
        if len(deck) == 0: #Every time the deck is empty, we get a new shuffled deck and reset the card count
            deck = DECK.copy()
            random.shuffle(deck)
            count = 0

        new_card = deck.pop()
        dealer_hand.append(new_card)
        val = CARD_VALUES[new_card.split()[0]]
        if 2 <= val <= 6:
            count += 1
        elif val == 10 or new_card.split()[0] == "A":
            count -= 1
    return calculate_hand_value(dealer_hand), count, deck

def get_dqn_state(player_hand, dealer_card, count):
    total = calculate_hand_value(player_hand)
    usable_ace = 1 if any(card.startswith("A") and total <= 21 for card in player_hand) else 0
    dealer_value = 11 if dealer_card.split()[0] == "A" else CARD_VALUES[dealer_card.split()[0]]
    count_norm = (count + 20) / 40 #actual count value

    return np.array([total / 21, dealer_value / 11, usable_ace, count_norm], dtype=np.float32)

def choose_dqn_action(epsilon, valid_actions, state, policy_net, device, mode):
    with torch.no_grad(): #Exploit only on valid actions
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        q_values = policy_net(state_tensor).cpu().numpy()[0]
        for i in range(len(q_values)):
            if i not in valid_actions: #If Double-down is not allowed, we give it's Q-value a value of -inf so it never gets chosen in the max
                q_values[i] = float('-inf')
        action = int(np.argmax(q_values))
    if mode == "training" and random.random() < epsilon: #Explore
        action = random.choice(valid_actions)

    return action

class DQN(nn.Module):
    def __init__(self, input_dim=4, output_dim=3):
        super(DQN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.net(x)

def step_blackjack_env(action, player_hand, dealer_hand, dealer_card, deck, count):
    bet = 1
    bust_penalty = 1.1
    done = False

    if action == 2: #Double-down
        bet *= 2
        new_card = deck.pop()
        player_hand.append(new_card)
        count = adjust_count(new_card, count)
        if calculate_hand_value(player_hand) > 21: #If we busted we lose the bet
            return -bet*bust_penalty, None, True, count, deck
        player_total = calculate_hand_value(player_hand)
        dealer_total, count, deck = play_dealer_card_counting(deck, dealer_hand, count)
        if dealer_total > 21 or player_total > dealer_total: #If we won after the dealer played, we win the bet
            return bet, None, True, count, deck
        elif player_total < dealer_total: #If we lost after the dealer played, we lose the bet
            return -bet, None, True, count, deck
        else: #If it's a draw
            return 0, None, True, count, deck

    elif action == 0: #Hit
        if len(deck) == 0:
            deck = DECK.copy()
            random.shuffle(deck)
            count = 0
        new_card = deck.pop()
        player_hand.append(new_card)
        count = adjust_count(new_card, count)
        if calculate_hand_value(player_hand) > 21:
            return -bet*bust_penalty, None, True, count, deck
        else:
            next_state = get_dqn_state(player_hand, dealer_card, count)
            return 0, next_state, False, count, deck

    else: #Stay
        player_total = calculate_hand_value(player_hand)
        dealer_total, count, deck = play_dealer_card_counting(deck, dealer_hand, count)
        if dealer_total > 21 or player_total > dealer_total:
            return bet, None, True, count, deck
        elif player_total < dealer_total:
            return -bet, None, True, count, deck
        else:
            return 0, None, True, count, deck

def train_step(batch, policy_net, target_net, optimizer, device):
    gamma = 1
    states, actions, rewards, next_states, dones = zip(*batch)
    states = torch.tensor(np.array(states), dtype=torch.float32).to(device)
    actions = torch.tensor(actions).unsqueeze(1).to(device)
    rewards = torch.tensor(rewards, dtype=torch.float32).unsqueeze(1).to(device)
    dones = torch.tensor(dones, dtype=torch.float32).unsqueeze(1).to(device)
    next_states = torch.tensor(
      np.array([s if s is not None else np.zeros(4) for s in next_states]),  #We have 4 features for every state
      dtype=torch.float32
    ).to(device)

    q_values = policy_net(states).gather(1, actions)
    next_q_values = target_net(next_states).max(1, keepdim=True)[0].detach()
    target_q = rewards + gamma * next_q_values * (1 - dones)

    loss = nn.MSELoss()(q_values, target_q)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

def train_dqn_blackjack(episodes=200_000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    policy_net = DQN().to(device)
    target_net = DQN().to(device)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()
    optimizer = optim.Adam(policy_net.parameters(), lr=1e-4)
    replay_buffer = deque(maxlen=200_000)

    epsilon = 1.0
    steps_done = 0
    deck = DECK.copy()
    random.shuffle(deck)
    count = 0

    for episode in range(episodes):
        if len(deck) < 10:
            deck = DECK.copy()
            random.shuffle(deck)
            count = 0

        player_hand = [deck.pop(), deck.pop()]
        dealer_hand = [deck.pop(), deck.pop()]
        dealer_card = dealer_hand[1]
        count = calculate_count(player_hand, dealer_card, count)

        state = get_dqn_state(player_hand, dealer_card, count)
        done = False

        if check_natural_blackjack(player_hand):
            action = 1 #Stay
            reward = 1.5 #in natural blackjack, the player gets 1.5x his initial bet and the dealer doesn't get to play
            if check_natural_blackjack(dealer_hand):
                reward = 0
                count = adjust_count(dealer_hand[0], count)

            next_state = None
            done = True

            replay_buffer.append((state, action, reward, next_state, done))

            if len(replay_buffer) >= 64:
                batch = random.sample(replay_buffer, 64)
                train_step(batch, policy_net, target_net, optimizer, device)

            steps_done += 1
            if steps_done % 1000 == 0:
                target_net.load_state_dict(policy_net.state_dict())

            epsilon = max(0.05, epsilon*0.999975)
            # Print progress every 10,000 episodes
            if episode % 10000 == 0 and episode > 0:
                print(f"Episode {episode} completed - Epsilon: {epsilon:.4f}")
            continue

        while not done:
            valid_actions = get_valid_actions(player_hand)

            action = choose_dqn_action(epsilon, valid_actions, state, policy_net, device, "training")

            reward, next_state, done, count, deck = step_blackjack_env(
                action, player_hand, dealer_hand, dealer_card, deck, count
            )

            replay_buffer.append((state, action, reward, next_state, done))
            state = next_state

            if len(replay_buffer) >= 64:
                batch = random.sample(replay_buffer, 64)
                train_step(batch, policy_net, target_net, optimizer, device)

            steps_done += 1
            if steps_done % 1000 == 0:
                target_net.load_state_dict(policy_net.state_dict())

        epsilon = max(0.05, epsilon*0.999975)

        if episode % 10000 == 0 and episode > 0:
            print(f"Episode {episode} completed - Epsilon: {epsilon:.4f}")

    print("Training complete.")
    return policy_net


def evaluate_dqn_policy(policy_net, num_games=100_000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    policy_net.eval()

    wins, draws, losses = 0, 0, 0
    balance = 0
    total_bet = 0
    deck = DECK.copy()
    random.shuffle(deck)
    count = 0

    for _ in range(num_games):
        initial_bet = 1
        total_bet += initial_bet
        if len(deck) < 10:
            deck = DECK.copy()
            random.shuffle(deck)
            count = 0

        player_hand = [deck.pop(), deck.pop()]
        dealer_hand = [deck.pop(), deck.pop()]
        dealer_card = dealer_hand[1]
        count = calculate_count(player_hand, dealer_card, count)

        state = get_dqn_state(player_hand, dealer_card, count)
        done = False

        if check_natural_blackjack(player_hand):
            if check_natural_blackjack(dealer_hand):
                count = adjust_count(dealer_hand[0], count)
                balance += initial_bet
                draws += 1
                continue
            balance += 1.5*initial_bet + initial_bet
            wins += 1
            continue

        while not done:
            valid_actions = get_valid_actions(player_hand)

            action = choose_dqn_action(0, valid_actions, state, policy_net, device, "evaluation")

            reward, next_state, done, count, deck = step_blackjack_env(
                action, player_hand, dealer_hand, dealer_card, deck, count
            )

            if next_state is not None:
                state = next_state

        if reward > 0:
            if reward == 2*initial_bet:
                total_bet += initial_bet
            balance += 2*reward
            wins += 1
        elif reward < 0:
            if reward <= -2*initial_bet:
                total_bet += initial_bet
            losses += 1
        else:
            balance += initial_bet
            draws += 1

    total = wins + draws + losses
    print(f"Out of {wins + losses + draws} games:")
    print(f"Wins:  {wins / total:.2%}")
    print(f"Draws: {draws / total:.2%}")
    print(f"Losses:{losses / total:.2%}")
    print(f"Balance: {int(balance)}")
    print(f"Total bet: {int(total_bet)}")
    print(f"Profit win ratio: {50 - (1 - balance / total_bet)*100:.2f}%")

def save_dqn_policy(model, path="blackjack_dqn_policy.pth"):
    torch.save(model.state_dict(), path)

def load_dqn_policy(path="blackjack_dqn_policy.pth"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = DQN().to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    return model

###################################################### BETTING AGENT TRAINING ###########################################

reward_table = defaultdict(list)  # keys: (count, bet), values: list of rewards

count_visits = defaultdict(int)
count_wins = defaultdict(int)

def update_count_stats(count_state, reward):
    count_visits[count_state] += 1
    if reward > 0:
        count_wins[count_state] += 1

def print_count_stats():
    print(f"{'Count':>6} | {'Visits':>6} | {'Win Ratio':>9}")
    print("-" * 26)
    for count_state in sorted(count_visits.keys()):
        visits = count_visits[count_state]
        wins = count_wins[count_state]
        win_ratio = wins / visits if visits > 0 else 0
        print(f"{count_state:>6} | {visits:>6} | {win_ratio:>9.2%}")

class BettingNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)  #Predict expected reward
        )

    def forward(self, x):
        return self.net(x)

def train_betting_agent(policy_net, episodes=100_000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    betting_model = BettingNetwork().to(device)
    optimizer = optim.Adam(betting_model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    replay_buffer = deque(maxlen=100_000)
    policy_net.eval()

    deck = DECK.copy()
    random.shuffle(deck)
    count = 0
    bet = 0
    initial_bet = 1
    beta = 2

    for episode in range(episodes):
        if len(deck) < 10:
            deck = DECK.copy()
            random.shuffle(deck)
            count = 0

        count_state = count
        advantage = 1 / (1 + np.exp(-0.5 * count_state)) #sigmoid function for player advantage based on count
        bet = random.randint(1, 10) #try out different bets

        player_hand = [deck.pop(), deck.pop()]
        dealer_hand = [deck.pop(), deck.pop()]
        dealer_card = dealer_hand[1]
        count = calculate_count(player_hand, dealer_card, count)

        state = get_dqn_state(player_hand, dealer_card, count)
        done = False

        if check_natural_blackjack(player_hand):
            action = 1 #Stay
            reward = bet
            if check_natural_blackjack(dealer_hand):
                reward = 0
                count = adjust_count(dealer_hand[0], count)
            next_state = None
            done = True

            key = (
                (count_state + 20) / 40,  # normalized count
                bet / 10,                 # normalized bet
                advantage,
            )
            value = reward - beta*(1-advantage)**2*bet #bias the value to make the network understand that higher counts means higher bet

            reward_table[key].append(value)

            update_count_stats(count_state, reward)

            continue

        while not done:
            valid_actions = get_valid_actions(player_hand)

            action = choose_dqn_action(0, valid_actions, state, policy_net, device, "evaluation")

            reward, next_state, done, count, deck = step_blackjack_env(
                action, player_hand, dealer_hand, dealer_card, deck, count
            )

            if next_state is not None:
                state = next_state

        if reward > 0:
            reward = bet
        elif reward < 0:
            reward = -bet
        else:
            reward = 0

        # Store transition
        key = (
            (count_state + 20) / 40,  # normalized count
            bet / 10,                 # normalized bet
            advantage,
        )
        value = reward - beta*(1-advantage)**2*bet

        reward_table[key].append(value)

        update_count_stats(count_state, reward)

        # Train the betting model
        if len(reward_table) >= 64:
            keys = random.sample(list(reward_table.keys()), 64)
            counts, bets, advantages = zip(*keys)
            rewards = [np.mean(reward_table[k]) for k in keys] #mean value so the target is more stable

            inputs = torch.tensor(np.column_stack((counts, bets, advantages)), dtype=torch.float32).to(device)
            targets = torch.tensor(rewards, dtype=torch.float32).unsqueeze(1).to(device)

            preds = betting_model(inputs)
            loss = loss_fn(preds, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if (episode + 1) % 1000 == 0:
            print(f"[Episode {episode + 1}] Loss: {loss.item():.4f}")

    return betting_model

def choose_best_bet(count, betting_model):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    min_bet=1
    max_bet=10
    advantage = 1 / (1 + np.exp(-0.5 * count))
    betting_model.eval()
    possible_bets = list(range(min_bet, max_bet + 1))
    inputs = torch.tensor(
        [[(count + 20) / 40, b / 10, advantage] for b in possible_bets],
        dtype=torch.float32
    ).to(device)
    with torch.no_grad():
        values = betting_model(inputs).squeeze().cpu().numpy()
    best_bet = possible_bets[np.argmax(values)]
    return best_bet

def evaluate_betting_policy(betting_model):
    min_bet=1
    max_bet=10
    betting_model.eval()
    for count in range(-20, 21):
        bets = list(range(min_bet, max_bet + 1))
        advantage = 1 / (1 + np.exp(-0.5 * count))
        inputs = torch.tensor(
            [[(count + 20) / 40, bet / 10, advantage] for bet in bets],
            dtype=torch.float32
        )
        with torch.no_grad():
            predicted_rewards = betting_model(inputs).squeeze()
            best_bet_idx = torch.argmax(predicted_rewards).item()
            best_bet = bets[best_bet_idx]
        print(f"Count {count: >3}: Bet {best_bet: >3}")

################################# EVALUATION OF BOTH AGENTS TOGETHER ############################
def evaluate_optimal_policy_with_betting(policy_net, betting_model, num_games=100_000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    policy_net.eval()
    betting_model.eval()

    wins, draws, losses = 0, 0, 0
    balance = 0
    total_bet = 0
    deck = DECK.copy()
    random.shuffle(deck)
    count = 0

    for _ in range(num_games):
        if len(deck) < 10:
            deck = DECK.copy()
            random.shuffle(deck)
            count = 0

        count_state = count
        bet = choose_best_bet(count_state, betting_model)

        initial_bet = 1

        player_hand = [deck.pop(), deck.pop()]
        dealer_hand = [deck.pop(), deck.pop()]
        dealer_card = dealer_hand[1]

        count = calculate_count(player_hand, dealer_card, count)

        state = get_dqn_state(player_hand, dealer_card, count)
        done = False

        if check_natural_blackjack(player_hand):
            total_bet += bet
            if check_natural_blackjack(dealer_hand):
                count = adjust_count(dealer_hand[0], count)
                balance += bet
                draws += 1

                update_count_stats(count_state, 0)

                continue

            update_count_stats(count_state, bet)

            balance += 1.5*bet + bet
            wins += 1
            continue

        while not done:
            valid_actions = get_valid_actions(player_hand)

            action = choose_dqn_action(0, valid_actions, state, policy_net, device, "evaluation")

            reward, next_state, done, count, deck = step_blackjack_env(
                action, player_hand, dealer_hand, dealer_card, deck, count
            )

            if next_state is not None:
                state = next_state

        if reward > 0:
            if reward == 2*initial_bet:
                total_bet += 2*bet
                balance += 4*bet
            else:
                total_bet += bet
                balance += 2*bet
            wins += 1
        elif reward < 0:
            if reward <= -2*initial_bet:
                total_bet += 2*bet
            else:
                total_bet += bet
            losses += 1
        else:
            total_bet += bet
            balance += bet
            draws += 1

        update_count_stats(count_state, reward)

    total = wins + draws + losses
    print(f"Out of {wins + losses + draws} games:")
    print(f"Wins:  {wins / total:.2%}")
    print(f"Draws: {draws / total:.2%}")
    print(f"Losses:{losses / total:.2%}")
    print(f"Balance: {int(balance)}")
    print(f"Total bet: {int(total_bet)}")
    print(f"Profit win ratio: {50 - (1 - balance / total_bet)*100:.2f}%")


if __name__ == "__main__":
    if os.path.exists("blackjack_dqn_policy.pth"):
        print("Loading saved DQN policy...")
        trained_policy = load_dqn_policy()
    else:
        print("Training new DQN policy...")
        trained_policy = train_dqn_blackjack()
        save_dqn_policy(trained_policy)

    betting_model = None

    while True:
        print("\n=== Blackjack System Menu ===")
        print("1. Evaluate the playing agent")
        print("2. Train the betting agent")
        print("3. Evaluate the trained betting agent")
        print("4. Evaluate the full system (playing + betting)")
        print("5. Exit")
        choice = input("Enter your choice: ")

        if choice == "1":
            evaluate_dqn_policy(trained_policy)

        elif choice == "2":
            print("Training betting strategy...")
            betting_model = train_betting_agent(trained_policy)
            torch.save(betting_model.state_dict(), "betting_model_regression.pth")
            print("Training finished and betting model saved as betting_model_regression.pth.")

        elif choice == "3":
            if betting_model is None:
                betting_model = BettingNetwork()
                if os.path.exists("betting_model_regression.pth"):
                    betting_model.load_state_dict(torch.load("betting_model_regression.pth"))
                    betting_model.eval()
                else:
                    print("No saved betting model found. Please train it first.")
                    continue
            evaluate_betting_policy(betting_model)
            print_count_stats()
            count_visits.clear()
            count_wins.clear()

        elif choice == "4":
            if betting_model is None:
                betting_model = BettingNetwork()
                if os.path.exists("betting_model_regression.pth"):
                    betting_model.load_state_dict(torch.load("betting_model_regression.pth"))
                    betting_model.eval()
                else:
                    print("No saved betting model found. Please train it first.")
                    continue
            evaluate_optimal_policy_with_betting(trained_policy, betting_model)
            print_count_stats()

        elif choice == "5":
            print("Exiting. Goodbye!")
            break

        else:
            print("Invalid choice. Please enter 1, 2, 3, 4 or 5.")

Episode 10000 completed - Epsilon: 0.7408
Episode 20000 completed - Epsilon: 0.5488
Episode 30000 completed - Epsilon: 0.4066
